# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Published:    {getattr(metadata, 'datePublished', '')}")
print(f"Description:  {getattr(metadata, 'description', '')}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Record sets are collections of tabular or semi-structured data in the Croissant schema. Each record set, field, and column has an `@id`. Let's display all available record sets and their metadata.

In [ ]:
# List record sets and their fields
if not hasattr(metadata, 'recordSet') or not metadata.recordSet:
    print("No record sets found in this dataset. Searching for files with tabular data...")
    # Try to get data file objects
    if hasattr(metadata, 'distribution') and metadata.distribution:
        for ds in metadata.distribution:
            contentUrl = getattr(ds, 'contentUrl', None)
            encodingFormat = getattr(ds, 'encodingFormat', None)
            name = getattr(ds, 'name', None)
            print(f"Distribution @id: {getattr(ds, '@id', None)} | Name: {name} | Format: {encodingFormat} | URL: {contentUrl}")
    else:
        print("No tabular data found in metadata file objects.")
else:
    for recset in metadata.recordSet:
        print(f"RecordSet @id: {getattr(recset, '@id', None)} | Name: {getattr(recset, 'name', None)}")
        if hasattr(recset, 'field'):
            for field in recset.field:
                print(f"  Field @id: {getattr(field, '@id', None)} | Name: {getattr(field, 'name', None)} | DataType: {getattr(field, 'dataType', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

From our exploration, this Croissant schema expresses its main table via a single distribution at:
* Distribution `@id`: `https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034`

We will attempt to load data from this table via mlcroissant.

In [ ]:
# The Croissant schema for this dataset (2024-06) does not explicitly list record sets, but uses standard DataFile distribution.
# By standard, `mlcroissant.Dataset.records()` accepts 'record_set' as either a @id or a table name or 'default'.

# Given the lack of explicit recordSet, try loading records using the known distribution @id.
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # The only distribution

try:
    records_iter = dataset.records(record_set=main_record_set_id)
    sample_rows = [r for i, r in enumerate(records_iter) if i < 5]
    pprint.pprint(sample_rows)
except Exception as e:
    print(f"Error loading records: {e}")

## 4. DataFrame Construction
Let's load all records from the (sole) main record set into a pandas DataFrame for analysis.

In [ ]:
# Only one main table identified
record_sets = [main_record_set_id]
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df

main_df = dataframes[main_record_set_id]
print(main_df.columns.tolist())
main_df.head()

## 5. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Let's analyze the 'MW [kDa]' (molecular weight) field if present. We'll:
- Filter proteins with MW greater than a threshold.
- Normalize the MW.
- Group by 'Description' and calculate mean MW.

In [ ]:
# Identify field names
df = main_df
df_columns_lower = [col.lower() for col in df.columns]

# Map relevant field names to variables, preferring Croissant @id naming
mw_field = None
for col in df.columns:
    if col.strip().lower() in ['mw [kda]', 'mw', 'molecularweight', 'molecular_weight', 'molecular weight']:
        mw_field = col
        break
if not mw_field:
    raise ValueError("Could not find molecular weight column in data.")

# Group field: Description or Protein name if available
group_field = None
for col in df.columns:
    if col.strip().lower() in ['description', 'protein name', 'name']:
        group_field = col
        break

# Filter records
threshold = 20  # kDa
filtered_df = df[df[mw_field].astype(float) > threshold]
print(f"Records with {mw_field} > {threshold}:")
print(filtered_df[[mw_field]].head())

# Normalize MW
filtered_df = filtered_df.copy()
mw_norm_col = f"{mw_field}_normalized"
filtered_df[mw_norm_col] = (filtered_df[mw_field].astype(float) - filtered_df[mw_field].astype(float).mean()) / filtered_df[mw_field].astype(float).std()
print(f"Normalized {mw_field} for filtered records:")
print(filtered_df[[mw_field, mw_norm_col]].head())

# Group and summarize
if group_field:
    grouped_df = filtered_df.groupby(group_field)[mw_field].mean().to_frame('Mean MW')
    print(f"Grouped data by {group_field}, mean MW:")
    print(grouped_df.head())

## 6. Visualization
Visualize the distribution of protein molecular weights and their relationships to other fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of MW
plt.figure(figsize=(9,5))
df[mw_field].astype(float).hist(bins=30)
plt.title(f"Distribution of {mw_field}")
plt.xlabel(mw_field)
plt.ylabel("Number of proteins")
plt.show()

# If abundance columns exist, plot scatter MW vs. abundance
abundance_col = None
for col in df.columns:
    if 'abundance' in col.lower() or ('norm' in col.lower() and 'abundance' in col.lower()):
        abundance_col = col
        break
if abundance_col:
    plt.figure(figsize=(8,5))
    plt.scatter(df[mw_field].astype(float), df[abundance_col].astype(float), alpha=0.6)
    plt.xlabel(mw_field)
    plt.ylabel(abundance_col)
    plt.title(f"{mw_field} vs {abundance_col}")
    plt.show()

## 7. Conclusion
This notebook demonstrates how to access FAIR-structured `Croissant` datasets with mlcroissant, extract and analyze molecular weight statistics on human protein mass spectrometry data, and visualize distributions. You can further extend this analysis by exploring other fields, filtering by peptide count, or replicating this workflow on other Croissant-compliant datasets.